In [34]:
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score


In [35]:
data = pd.read_csv("/Users/codyrlee/Documents/CodyLee_CapstoneProject/clean_data/churn_model_data.csv")

data = data[data['deal_status'] == 'Closed -Won']

In [36]:
# KDA 

data = pd.get_dummies(data, columns=['feature_purchased', 'industry', 'region'], drop_first=True)

X = data[['client_tenure_months', 'closed_won_usd', 'sessions_last_30d', 'csat_score', 'comment_sentiment_score', 'support_tickets_30d', 'last_login_days'] + [c for c in data.columns if c.startswith('industry') or c.startswith('feature_purchased') or c.startswith('region')]]
y = data['churn']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Get feature importances
importances = rf_model.feature_importances_

# Create a Series for better readability
feature_names = X.columns
feature_importance_series = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print("Feature Importances:")
print(feature_importance_series)

Feature Importances:
client_tenure_months                 0.454920
sessions_last_30d                    0.083414
closed_won_usd                       0.079461
csat_score                           0.069366
last_login_days                      0.065888
support_tickets_30d                  0.063671
comment_sentiment_score              0.061466
feature_purchased_Core               0.013423
industry_HealthTech                  0.010110
region_LATAM                         0.009905
industry_MarTech                     0.009839
region_Europe                        0.009619
region_North America                 0.009371
feature_purchased_Analytics          0.008878
feature_purchased_Premium Support    0.008675
industry_E -Commerce                 0.008565
industry_SMB SaaS                    0.008191
industry_Logistics                   0.007522
industry_EdTech                      0.006872
feature_purchased_Automation         0.006825
industry_FinTech                     0.004018
dtype: float6

In [ ]:

# Generate churn probabilities
churn_probabilities = rf_model.predict_proba(X_test)

# The second column (index 1) contains the probability of churn (class 1)
probability_of_churn = churn_probabilities[:, 1]

print("Churn Probabilities for the test set:")
print(probability_of_churn)

# Optional: Evaluate the model's performance (e.g., using AUC-ROC)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
auc_roc = roc_auc_score(y_test, y_pred_proba)
print(f"\nAUC-ROC Score: {auc_roc:.2f}")

Churn Probabilities for the test set:
[0.96 1.   1.   0.98 0.73 0.99 0.99 0.99 0.99 0.99 0.95 0.96 0.99 1.
 0.98 0.47 0.97 0.99 0.98 1.   0.98 0.98 1.   1.   0.95 1.   0.98 0.98
 0.97 0.94 0.97 0.98 1.   0.97 1.   1.   1.   1.   1.   0.97 0.97 1.
 1.   0.99 1.   1.   0.97 0.97 0.99 0.99 1.   0.48 0.99 0.99 0.99 1.
 0.98 1.   0.97 0.5  0.99 0.96 1.   0.98 1.   0.58 0.97 1.   0.95 1.
 0.99 0.48 0.99 1.   1.   0.93 0.96 0.97 1.   0.99 0.99 0.97 1.   0.49
 0.44 0.99 0.51 1.   0.99 0.97 0.99 0.88 1.   0.96 0.85 1.   1.   0.99
 0.94 1.   0.94 0.99 1.   0.77 1.   0.99 0.99 0.99 0.96 1.   0.99 0.99
 0.8  1.   0.91 1.   0.98 0.97 0.97 0.96 1.   0.99 1.   0.99 1.   0.99
 0.99 1.   1.   1.   0.91 0.99 0.93 1.   0.99 0.86 1.   1.   1.   1.
 0.99 0.99 0.44 1.   0.93 1.   1.   0.97 1.   1.   0.86 1.   1.   0.96
 0.93 0.96 1.   0.99 0.99 1.   1.   1.   0.99 1.   1.   1.   0.51 1.
 0.99 1.   1.   1.   1.   0.97 0.98 0.97 1.   1.   1.   0.98 0.96 0.84
 1.   0.99 1.   1.   0.98 0.93 1.   1.   0.98 1.   